# Bloom Filter - Demonstration and Testing

This notebook demonstrates the usage of our custom `BloomFilter` data structure implemented in `bloom_filter.py`.

According to the project requirements, this notebook serves two main purposes:
1. **Demonstration:** Showing basic insertions, membership checks, and explaining the probabilistic nature of Bloom Filters.
2. **Correctness & Hash Testing:** Rigorously testing the filter to ensure zero false negatives, verifying that the empirical false positive rate matches the target rate, and proving the uniform distribution of our custom hash functions.

In [ ]:
!pip install matplotlib

In [ ]:
import random
import matplotlib.pyplot as plt
from bloom_filter import BloomFilter

print("Bloom Filter module and dependencies loaded successfully!")

## 1. Basic Demonstration & Understanding False Positives
A Bloom filter never produces **false negatives** (if an item was added, it will definitely return `True`). However, it can produce **false positives** (an item not added might still return `True` by coincidence).

Let's demonstrate this concept.

In [ ]:
# Initialize the Bloom Filter for a small dataset (20 items) with a 5% false positive rate
bf_demo = BloomFilter(expected_items=20, fp_rate=0.05)

fruits_to_add = ["apple", "banana", "cherry", "strawberry", "orange"]

print("--- Inserting Items into Bloom Filter ---")
for fruit in fruits_to_add:
    bf_demo.add(fruit)
    print(f"Inserted: {fruit}")

print("\n--- Checking Membership (Added Items) ---")
print(f"Is 'apple' in the filter?      -> {bf_demo.check('apple')} (Expected: True)")
print(f"Is 'strawberry' in the filter? -> {bf_demo.check('strawberry')} (Expected: True)")

print("\n--- Checking Non-Existing Items ---")
# Not: Bu öğeler eklenmediği için genellikle False döner, 
# ancak %5'lik bir ihtimalle True (False Positive) dönebilir.
print(f"Is 'grape' in the filter?      -> {bf_demo.check('grape')} (Expected: Mostly False, rarely True)")
print(f"Is 'watermelon' in the filter? -> {bf_demo.check('watermelon')} (Expected: Mostly False, rarely True)")

## 2. Correctness Testing: Zero False Negatives Guarantee
The most critical property of a Bloom Filter is that it must return `True` for every item that has been inserted. We will insert 10,000 items and assert that checking them yields a 100% success rate.

In [ ]:
N_ITEMS = 10000
bf_correctness = BloomFilter(expected_items=N_ITEMS, fp_rate=0.05)

# Generate sample data
added_data = [f"item_{i}" for i in range(N_ITEMS)]

# Insert all items
for item in added_data:
    bf_correctness.add(item)

# Test for false negatives
false_negatives = 0
for item in added_data:
    if not bf_correctness.check(item):
        false_negatives += 1

assert false_negatives == 0, f"Error: Found {false_negatives} false negatives!"
print("Success! 0% False Negative Rate confirmed. All inserted items were found.")

## 3. Correctness Testing: Empirical False Positive Rate
We designed our filter to have a False Positive Rate (FPR) of 5% (0.05). Let's query the filter with 50,000 completely new items that were *never* inserted, and calculate the actual empirical FPR to see if it aligns with our target.

In [ ]:
N_TEST = 50000
test_data = [f"never_added_{i}" for i in range(N_TEST)]

false_positives = sum(1 for item in test_data if bf_correctness.check(item))
empirical_fpr = false_positives / N_TEST

print(f"Target False Positive Rate   : {bf_correctness.fp_rate:.4f}")
print(f"Empirical False Positive Rate: {empirical_fpr:.4f}")

import math
assert math.isclose(empirical_fpr, bf_correctness.fp_rate, abs_tol=0.02), "Empirical FPR is too far from target!"
print("Success! The empirical FPR is closely matching the theoretical target.")

## 4. Hash Function Uniformity Test
A good Bloom Filter relies on hash functions that distribute indices uniformly across the bit array. If the indices cluster in specific regions, the False Positive rate will skyrocket.

Here, we will generate thousands of items, pass them through our `_get_hash_indices` method (which uses salted `hashlib.md5`), and plot a histogram of the resulting bit array indices. A flat, uniform histogram proves our hash function is performing correctly.

In [ ]:
# Create a filter with a specific bit array size
bf_hash_test = BloomFilter(expected_items=5000, fp_rate=0.05)

all_indices = []
# Generate 10,000 random items to hash
for i in range(10000):
    dummy_item = f"hash_test_string_{i}"
    # Retrieve the 'k' indices for this item
    indices = bf_hash_test._get_hash_indices(dummy_item)
    all_indices.extend(indices)

# Plotting the distribution
plt.figure(figsize=(10, 5))
plt.hist(all_indices, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
plt.axhline(y=(len(all_indices)/50), color='r', linestyle='dashed', linewidth=1.5, label='Expected Uniform Frequency')
plt.title(f"Hash Function Index Distribution (Bit Array Size: {bf_hash_test.size})")
plt.xlabel("Bit Array Index")
plt.ylabel("Frequency")
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

print("As seen in the histogram, the generated hash indices are uniformly distributed across the entire bit array.")